# Lecture 4 Tutorial: Working with clinical text


## What we will cover

| Part | Topic | Time |
|------|-------|------|
| 0 | Setup, install spaCy | 2 min |
| 1 | Load & explore the data | 4 min |
| 2 | Data quality  | 7 min |
| 3 | Work with dates| 4 min |
| 4 | Extract structure from free text  | 5 min |
| 5 | NLP with spaCy & negspaCy | 10 min |
| ⭐ Advanced | Entity extraction & evaluation | if time allows |

<font color="red">The cells below red text are part of exercises and may not run properly without inputting further code. </font>

<font color="green">If you prefer to just run the cells and follow what is happening in the exercises, click the toggle below green text to reveal the correct code. </font>

---
## Part 0: Setup

It may take a minute to download the spaCy language model, best to be done before the tutorial.
While it downloads, read ahead to Part 1.

In [ ]:
#Calls already pre-installed modules needed to install other modules within Colab using python
import subprocess, sys

#Installs spacy, negspacy, and gdown in Colab so we can call it
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy', 'negspacy', 'gdown'], check=True)

#Instructs the space module to download en_core_web_sm
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '-q'], check=True)

#Calls various pre-installed and newly installed modules needed for the tutorial
import pandas as pd
import re
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import spacy
from negspacy.negation import Negex

#Loads the downloaded en_core_web_sm model and calls it 'nlp'
nlp = spacy.load('en_core_web_sm')
print('spaCy', spacy.__version__, '— model loaded.')

---
## Part 1: Load & explore the data

Run the code block below to download the dataset into Google Colab.


In [ ]:
import gdown
from pathlib import Path

Path('./data').mkdir(exist_ok=True) # Ensures the content folder exists for the dataset to be downloaded into
DATASET_DIR = Path("./data") # Path of the extracted dataset.
DATASET_FILE = Path("./data/synthetic_hand_wrist_xray_reports_v2.csv") # Name of the extracted dataset.

if not DATASET_FILE.exists(): # Checks if the dataset is already downloaded in Colab
    gdown.download(id="1ZcHcVlTA8px3rox7cMRSjoUVV4zzX03I", output=str(DATASET_FILE), quiet=False) #Downloads the dataset from a public Google Drive link
    print(f"Dataset ready at {DATASET_DIR}")
else:
    print(f"Dataset already exists at {DATASET_DIR} - skipping download.")

Our dataset has four columns:

| Column | Description |
|--------|-------------|
| `fake_patient_id` | De-identified patient ID |
| `scan_datetime` | When imaging was performed |
| `report_datetime` | When the radiologist signed off |
| `report` | Free-text radiology report |

Each report is semi-structured: an exam header, then labelled sections
(`Indications:`, `Findings:`, `Impression:` / `Conclusion:`).

First, let's load our downloaded dataset, find out how many reports exist in it, and how many different patients these reports are for. We will also preview the first 3 rows of the dataset.

In [ ]:
df = pd.read_csv('data/synthetic_hand_wrist_xray_reports_v2.csv') #Loads a csv file as a pandas dataframe
print(f'{len(df)} rows, {df["fake_patient_id"].nunique()} unique patients') #Outputs the number of rows of the dataframe, and the number of unique values in the patient_id column
df.head(3) #Outputs the top 3 rows of the dataframe

Let's pretty-print two reports to see its structure.

In [ ]:
#Makes a function that inputs the text of a report, splits it up by the different headings, and prints the first report in our dataframe
def pretty_print(report): 
    formatted = re.sub(
        r'\s*(Indications?|Findings|Impressions?|Conclusions?|Reported By)\s*:',
        r'\n\n\1:',
        report
    )
    print(formatted.strip())

pretty_print(df.loc[0, 'report'])  

In [ ]:
#Uses the function we just made to print the second report in our dataframe
pretty_print(df.loc[1, 'report'])  # Boxer's fracture

### Exercise 1.1. Quick exploration

1. What date range do the scans cover?
2. How many reports contain the word **"fracture"** anywhere?

<font color="red">If you wish to code, this cell contains helpful code structures you can use to obtain the answer:</font>

In [ ]:
df['report_datetime'] = pd.to_datetime(df['report_datetime']) #Converts the inputted datetime column to a format which can be further manipulated. You will have to do this for the scan date-time column.
print(df['report_datetime'].min().date()) #Prints the earliest date where a report was written. You will need to find the earliest and latest scan dates.

dislocation_count = df['report'].str.contains('dislocation', case=False).sum()
print(dislocation_count) #Goes through each report in the dataset and checks if it contains the word 'dislocation', independent of case, then sums up how many actually did. You will need to do the same for the word 'fracture'


<font color="green">Example code to obtain the answers is below:</font>

In [ ]:
#@title Click to reveal code
df['scan_datetime'] = pd.to_datetime(df['scan_datetime']) #Converts the inputted datetime column to a format which can be further manipulated.
print('Date range:', df['scan_datetime'].min().date(), 'to', df['scan_datetime'].max().date()) #Finds the earliest and latest dates where a scan was performed.

fracture_count = df['report'].str.contains('fracture', case=False).sum() #Goes through each report in the dataset and checks if it contains the word 'fracture', independent of case, then sums up how many actually did.
print(f'Reports containing "fracture": {fracture_count} / {len(df)}') #Prints how many reports contain the word fracture, out of how many reports there are total.

---
## Part 2: Data quality —  Duplicates

Real EHR systems often store the same report more than once.
We identified three patterns during the lecture:

| Type | What it looks like | Why it matters |
|------|--------------------|----------------|
| **Exact duplicate** | Identical text, same patient, same accession | Inflates counts, causes data leakage in train/test splits |
| **Near-duplicate** | Same text with a trailing space, period, or encoding artefact | Won't be caught by exact-match deduplication |
| **Addendum** | A corrected version stored as a new row days later | If you keep the original you may train on the wrong label |

### 2.1. Identify patients with more than one row

Assuming every patient only has a single patient ID, all the patterns of duplicates should be caught by them having the same patient ID as each other, so that is where we will start in trying to identify the duplicate reports.

In [ ]:
patient_counts = df['fake_patient_id'].value_counts() #Counts how many times each value in the patient_id column appears, creating a table with each patient ID and the count of how many times they appear
print(f'Total rows: {len(df)}') #Finds how many rows are in the dataframe
print(f'Unique patients: {len(patient_counts)}') #Finds how many different patient ids there are
print()
print('Patients with more than one row:') 
print(patient_counts[patient_counts > 1]) #Filters the table created above to only include the patients where the count is more than 1.

Let's view the start of the reports for these patients to try and manually see if they are duplicates, and if so, what pattern of duplicates they are.

In [ ]:
# Print all rows for each duplicated patient
dup_patients = patient_counts[patient_counts > 1].index.tolist() #Extract the patient IDs that appear more than once as a Python list

for pid in dup_patients: #Loop through each duplicate patient ID one by one
    print(f'\n=== {pid} ===') #Prints a clear section header for the current duplicate patient
    for _, row in df[df['fake_patient_id'] == pid].iterrows(): #Filters dataframe for rows matching the current patient ID and loop through them
        print(f'  report_datetime: {row["report_datetime"]}') #Print the date-time of the report
        print(f'  report start: {row["report"][:120]}...') #Print the first 120 characters of the report content
        print()

### 2.2. Step 1: Handle addendums

Addendums can appear differently depending on your EHR.
In this tutorial, addendum rows start with (or contains) the word **"Addendum"**. First, let's find the reports which contain addendums.

In [ ]:
# Flag addendum rows
df['is_addendum'] = df['report'].str.contains(r'\baddendum\b', case=False, na=False) #Create a true/false flag column; marks True if the report contains the exact word "addendum" 
print('Addendum rows found:') 
print(df[df['is_addendum']][['fake_patient_id', 'report_datetime', 'report']].to_string()) #Filter and print only the rows flagged as addendums

Now for each patient who has an addendum, we will drop all their non-addendum rows.

In [ ]:
patients_with_addendum = df[df['is_addendum']]['fake_patient_id'].unique() #Extracts unique list of patient IDs that have at least one addendum row

#Creates a logic mask to drop non-addendum records for patients who DO have an addendum
keep_mask = ~(
    df['fake_patient_id'].isin(patients_with_addendum) & ~df['is_addendum']
)
#Apply the mask to filter the dataframe, remove the temporary flag column created earlier, and reset row numbers
df = df[keep_mask].drop(columns='is_addendum').reset_index(drop=True)

#Prints the final row count to verify how many records remain after cleanup
print(f'After addendum cleanup: {len(df)} rows')

### Exercise 2.3. 
Whilst the code above works fine to identify addendums and remove all the non-addendum rows in our dataset, can you think of reasons why this code may not work when dealing with 1000s of reports of real patients?

<details>
<summary><b>Click to show answer</b></summary>
1. When extracting addendum rows, we just look for the word 'addendum', anywhere in the report. Radiology reports may have a phrase in the text saying "please wait for consultant addendum", which would be picked up incorrectly as being a report that contains an addendum.
<br>
2. Imagine a patient had one scan, which had an initial report then the same report with an addendum, then had a different scan a year later, but only one report for this. Our code only distinguishes between different patients and not different scans, so the report for the second scan would incorrectly be dropped.
</details>


### 2.4. Step 2: Remove exact and near-exact duplicates

After handling addendums, we still have patients with multiple identical (or nearly identical) rows.

**Normalise first**, then deduplicate:
1. Strip leading/trailing whitespace
2. Collapse any run of whitespace to a single space
3. Remove trailing punctuation or spaces 

> **Note:** These normalisation steps are only used to identify duplicates.



In [ ]:
#Earlier, we converted the scan datetime to a date-time format for further manipulation in python. Here we do the same for the report datetime.
df['report_datetime'] = pd.to_datetime(df['report_datetime'])

#Define a cleaning function to clean and standardise the report text
def normalise(text):
    if not isinstance(text, str): #If the text is missing or not a string, return an empty string to avoid errors
        return ''
    text = text.strip() # Remove leading and trailing spaces from the text
    text = re.sub(r'\s+', ' ', text) #Collapse multiple consecutive spaces/newlines into a single space
    text = re.sub(r'[\s.]+$', '', text) #Strip away any spaces or full stops remaining at the very end of the text
    return text

#Apply the cleaning function to the 'report' column and save it as a new column
df['report_norm'] = df['report'].apply(normalise)

df_sorted = df.sort_values('report_datetime') #Sort the dataframe by the report date-time

dup_mask = df_sorted.duplicated(subset=['fake_patient_id', 'report_norm'], keep='last') # Flag any exact duplicates when looking at the patient ID and cleaned report and keep the last (latest report_datetime if they differ)

#Print the total count and previews of the duplicate rows scheduled for removal
print(f'Duplicate rows identified: {dup_mask.sum()}')
print()
print('Rows that will be dropped:')
print(df_sorted[dup_mask][['fake_patient_id', 'report_datetime', 'report']].to_string())

Now we drop the duplicate rows marked for removal and tidy up the dataset

In [ ]:
df = (df_sorted[~dup_mask] #Invert the duplicate flag to keep only the rows which are NOT duplicates.
      .drop(columns='report_norm') #Remove the temporary cleaned text column to keep the dataframe clean
      .reset_index(drop=True)) #Reset the row numbering with the rows remaining

print(f'Clean dataset: {len(df)} rows, {df["fake_patient_id"].nunique()} unique patients')

---
## Part 3: Turnaround Time

**Turnaround time** (scan to signed report) is a key radiology performance metric.
Let's calculate it.

In [ ]:
df['turnaround_hours'] = (
    df['report_datetime'] - df['scan_datetime']
).dt.total_seconds() / 3600 #Get the difference of the datetime between scan and report, convert it to seconds, then convert it to hours - and report this as a new column

df[['fake_patient_id', 'scan_datetime', 'report_datetime', 'turnaround_hours']].head(5) #Shows the first 5 reports and their turnaround time

Now let's create a histogram to look at the distribution of turnaround times.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df['turnaround_hours'], bins=15, edgecolor='white', color='steelblue')
ax.set_xlabel('Turnaround time (hours)')
ax.set_ylabel('Number of reports')
ax.set_title('Scan to report turnaround time')
plt.tight_layout()
plt.show()
print(df['turnaround_hours'].describe().round(1))

### Exercise 3.1. Fastest and slowest

Find the patient IDs with the **shortest** and **longest** turnaround times.

<font color="red">If you wish to code, the following cell contains helpful code structures you can use to obtain the answer:</font>


In [ ]:
earliest_datetime = df['scan_datetime'].min() #Selects the earliest scan time. The min() works for number as well as date-time formats
earliest_id = df.loc[df['scan_datetime'].idxmin(), 'fake_patient_id'] #Selects the index of the row with the earliest scan-time, and returns the patient ID of that row

<font color="green">Example code to obtain the answers is below:</font>

In [ ]:
#@title Click to reveal code
fastest_id  = df.loc[df['turnaround_hours'].idxmin(), 'fake_patient_id'] #Find the patient ID with the shortest turnaround time using the index of the minimum value
fastest_hrs = df['turnaround_hours'].min() #Get the actual minimum value of turnaround hours in the dataset
slowest_id  = df.loc[df['turnaround_hours'].idxmax(), 'fake_patient_id'] #Find the patient ID with the longest turnaround time using the index of the minimum value
slowest_hrs = df['turnaround_hours'].max() #Get the actual maximum value of turnaround hours in the dataset

print(f'Fastest: {fastest_id}  ({fastest_hrs:.1f} h)')
print(f'Slowest: {slowest_id}  ({slowest_hrs:.1f} h)')

---
## Part 4: Extracting structure from text 

Each report has three logical sections:

```
Indications: clinical question, why the scan was requested
Findings: what the radiologist observed
Impression/
Conclusion: summary, diagnosis
```

We can split these with a **regular expression** a pattern-matching mini-language
built into Python's `re` module.

In [ ]:
#Creates a function which inputs a single report and splits it into the component sections
def extract_sections(report):
    HEADERS = r'(Indications?|Findings|Impressions?|Conclusions?|Reported By)' #Defines the regular expression pattern to catch common clinical report headers
    splits = [
        (m.group(1), m.start(), m.end()) #stores a list of the matching headers
        for m in re.finditer(HEADERS + r'\s*:', report, re.IGNORECASE) #Find all matching headers followed by a colon (e.g "Findings:") ignoring case
    ]
    sections = {} #Start off with a blank report - we are using a dictionary to store each of the sections

    sections['header'] = report[:splits[0][1]].strip() if splits else report #Extract any text that appears before the very first discovered header as the 'header' section

    for i, (name, _, end) in enumerate(splits): # Loop through each discovered report header to slice out its corresponding text section
        next_start = splits[i + 1][1] if i + 1 < len(splits) else len(report) #The next section starts where the next header begins, or at the very end of the report text
        content = report[end:next_start].strip() #Extract the text content sitting between the current header's end and the next header's start

        #Map plural, singular, or alternative header names into standard dictionary keys
        KEY_MAP = {
            'indication': 'indication', 'indications': 'indication',
            'findings': 'findings',
            'impression': 'impression', 'impressions': 'impression',
            'conclusion': 'impression', 'conclusions': 'impression',
            'reported by': 'reported_by',
        }

        #Map any of the alternatively spelt/used header name to the standardised version
        sections[KEY_MAP.get(name.lower(), name.lower())] = content
    return sections

#Test the function on the first row of the dataset and print the extracted keys and text blocks
for k, v in extract_sections(df.loc[0, 'report']).items():
    print(f'[{k}]\n{v}\n')

Now we have a function that works for a single report, let's split all the reports in the dataset into their component sections.

In [ ]:
parsed = df['report'].apply(extract_sections).apply(pd.Series) #Run the extraction function on every report, and expand the resulting dictionaries into separate columns
df = pd.concat([df, parsed], axis=1) #Merge the original dataframe and the new parsed columns together side-by-side
df[['fake_patient_id', 'indication', 'findings', 'impression']].head(3) #Show the first 3 rows of the dataset and their newly extracted sections

### Exercise 4.1

The indication almost always contains *"?fracture"* (the clinical question) which is
**not** a confirmed finding. How many reports contain the word fracture in the 'indication' section, and how many contain the word fracture in the 'findings' section?

<font color="red">If you wish to code, the following cell contains helpful code structures you can use to obtain the answer:</font>

In [ ]:
oa_in_impression   = df['impression'].str.contains('osteoarthritis', case=False, na=False).sum() #This looks in the 'impression' section, finds all the reports that contain the word 'osteoarthritis', and sums up how many they are.
print(oa_in_impression)

<font color="green">Example code to obtain the answers is below:</font>

In [ ]:
#@title Click to reveal code
in_indication = df['indication'].str.contains('fracture', case=False, na=False).sum() #This looks in the 'indication' section, finds all the reports that contain the word 'fracture', and sums up how many they are.
in_findings   = df['findings'].str.contains('fracture', case=False, na=False).sum() #This looks in the 'findings' section, finds all the reports that contain the word 'fracture', and sums up how many they are.

print(f'"fracture" in Indication : {in_indication}')
print(f'"fracture" in Findings   : {in_findings}')

---
## Part 5: NLP with spaCy & negspaCy

Regex is fast but fragile. Clinical NLP libraries give us proper language understanding.

**spaCy** runs a full Natural Language Processing (NLP) pipeline on text, including steps such as 
tokeniser, sentence splitter, part of speech tagger. You will learn more about it during Day 4.

**negspaCy** adds negation detection on top, using the dependency parse to tell
whether an entity is grammatically negated.

### 5.1. spaCy basics: tokenisation and sentences

Here, we explore two handy functions of spaCy. Tokenisation splits the report section into individual 'tokens', often individual words or punctuation, but also assigns contextual features to them. Sentence splitting identifies the individual sentences in the report section.

In [ ]:
sample_text = df.loc[0, 'findings']  #Grab the text from the 'findings' section of the first patient row
doc = nlp(sample_text) #Process the raw text through the spaCy pipeline

print('=== Tokens ===')
for token in doc[:12]: #Loop through the first 12 individual tokens in the document
    print(f'  {token.text!r:20s}  pos={token.pos_:6s}  dep={token.dep_}')  #For each token, print the raw text, the Part-of-Speech tag, and dependency label - these are different contextual features. 

print()
print('=== Sentences ===')
for sent in doc.sents: #Use spaCy's sentence splitter to loop through and print every distinct sentence found in the text
    print(' ', sent.text)

### 5.2. negspaCy: detecting negated entities

As well as finding tokens and sentences, spaCy can also find 'entities', which are specific words that have labels. It contains some default entities, such as place names, dates, or ordinals (in the example below, 'fifth' is picked up as a default entity), but we can add also add our own. Here we add the word 'Fracture/fracture' as an entity to find, and label it as a `FINDING`.
<br>
<br>
negspaCy is a plugin for spaCy that provides a negative component finder. This can be directed to look only at entities which we have labelled as a 'FINDING', looking around the entity to see if it is grammatically negated.

In [ ]:
if 'entity_ruler' not in nlp.pipe_names:
    ruler = nlp.add_pipe('entity_ruler') #Adds the entity finding tool from spaCy 
    ruler.add_patterns([ #Adds two new entities for the entity finding tool to find - fracture/Fracture - both of which we give the label of 'FINDING' to
        {'label': 'FINDING', 'pattern': 'fracture'},
        {'label': 'FINDING', 'pattern': 'Fracture'},
    ])

if 'negex' not in nlp.pipe_names:
    nlp.add_pipe('negex', config={'ent_types': ['FINDING']}) #Adds the negation finding tool from negspaCy, specifying it to only work on entities labelled 'FINDING'

#Create a list of mock clinical sentences to test both positive and negated scenarios
test_sentences = [
    'No acute fracture or dislocation.',
    'Acute nondisplaced fracture of the scaphoid waist.',
    'No acute fracture. Boxer fracture of the fifth metacarpal neck.',
]

for sent in test_sentences: #Process each sentence input
    doc = nlp(sent)
    for ent in doc.ents: #For each input sentence, loop through any discovered entities.
        print(f'  "{sent}"') #Print the input sentence which contains the entity
        print(f'  -> entity="{ent.text}"  negated={ent._.negex}') #Print the entity itself, then print if the negation finding tool found a negation for that entity.
        print()

### Exercise 5.3. Apply to the whole dataset

Use the entity and negation finding tools to classify every report in the dataset.
A report is **positive** if it contains at least one *non-negated* `FINDING` entity in the findings.

How many reports contain a fracture?

<font color="red">If you wish to code, the following cell contains helpful code structures you can use to obtain the answer:</font>

In [ ]:
#There are different ways to approach this task, but we suggest creating a function which inputs the findings column, and outputs TRUE if any fracture entity is found which is not negated, else returns FALSE

#To help you, we provide a function that looks at any ordinal entities, then outputs TRUE if any of them are not negated, before applying it to the 'impression' column. 
def spacy_has_ordinal(impression):
    if not isinstance(impression, str): #Return False immediately if the impressions text is missing or not a valid string
        return False
    doc = nlp(impression) #Process the text through spaCy
    return any(ent.label_ == 'ORDINAL' and not ent._.negex for ent in doc.ents) #Return True if any discovered entity is tagged as a 'ORDINAL' AND is not negated by negex

df['example_ordinals'] = df['impression'].apply(spacy_has_ordinal) #Applies the function to the entire findings column of our dataset
print(f'entity-negspaCy pipeline: {df["example_ordinals"].sum()} contain an ordinal which is not negated') #Sums up how many positive reports there are as per the entity-negspaCy pipeline

<font color="green">Example code to obtain the answers is below. Please run this block even if you found the answer yourself as the function is needed in the following part of the tutorial.</font>

In [ ]:
#@title Click to reveal code
#Create a function that says TRUE if any fracture entity is found which is not negated, else returns FALSE
def spacy_has_fracture(findings):
    if not isinstance(findings, str): #Return False immediately if the findings text is missing or not a valid string
        return False
    doc = nlp(findings) #Process the text through spaCy
    return any(ent.label_ == 'FINDING' and not ent._.negex for ent in doc.ents) #Return True if any discovered entity is tagged as a 'FINDING' AND is NOT negated by negex

df['spacy_positive'] = df['findings'].apply(spacy_has_fracture) #Applies the function to the entire findings column of our dataset
print(f'entity-negspaCy pipeline: {df["spacy_positive"].sum()} positive') #Sums up how many positive reports there are as per the entity-negspaCy pipeline


Let's compare that to our initial search strategy of just finding the reports which contained the word 'fracture', and see which reports contained the word 'fracture' where it was negated every time.

In [ ]:
naive = df['findings'].str.contains('fracture', case=False, na=False) #Code we executed earlier which just looked for the word 'fracture' in the findings section
print(f'Naive (keyword only): {naive.sum()} positive')
print()
print('Turned negative (were false positives):')
for _, row in df[naive & ~df['spacy_positive']].iterrows(): #Filter the dataframe for rows that matched the naive search but were marked negative by our pipeline
    print(f"  {row['fake_patient_id']}: {row['findings'][:100]}...") #Print the patient ID and the first 100 characters of the findings section to review the false positives

---
## Part 6: Handling uncertainty

Some reports neither confirm nor deny a fracture:
- *"fracture cannot be entirely excluded"*
- *"partly obscured by positioning"*
- *"slight cortical irregularity ... possibly projectional"*

We flag these as `uncertain` rather than lumping them with negatives.

In [ ]:
#Compile a case-insensitive regular expression pattern to detect clinical uncertainty phrases
UNCERTAINTY_PATTERNS = re.compile(
    r'cannot be (entirely |radiographically )?excluded'
    r'|cannot be (fully )?visualised'
    r'|partly obscured'
    r'|(?:subtle|slight|faint) (?:linear )?(?:lucency|density|irregularity|cortical)'
    r'|not entirely normal',
    re.IGNORECASE
)

#Define a function to categorize each row into 'uncertain', 'positive', or 'negative'
def classify_report(row) -> str:
    findings  = row['findings']  if isinstance(row['findings'],  str) else '' #Extract text from the findings section, defaulting to an empty string if missing
    impression = row['impression'] if isinstance(row['impression'], str) else '' #Extract text from the impression section, defaulting to an empty string if missing
    combined  = findings + ' ' + impression #Combine them together
    if UNCERTAINTY_PATTERNS.search(combined):  #Priority 1: If any uncertainty phrases match the text, classify the report as 'uncertain'
        return 'uncertain'
    if row['spacy_positive']: #Priority 2: If no uncertainty is found, check if our entity-negspaCy pipeline flagged a positive entity
        return 'positive'
    return 'negative' #Priority 3: Default to negative otherwise

df['classification'] = df.apply(classify_report, axis=1) #Apply the classification function across the dataframe row-by-row 
df['classification'].value_counts() #Count how many reports are in each classification category

Let's visualise the counts of reports in each category in a bar chart.

In [ ]:
counts = df['classification'].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(counts.index, counts.values, color=['#e74c3c', '#f39c12', '#2ecc71'])
ax.bar_label(bars)
ax.set_title('Report classification\n(section-aware + negspaCy)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

### What we built

```
raw CSV (53 rows, some duplicated)
  - addendum detection     
  - text normalisation        
  - exact deduplication       

clean CSV (50 rows, 1 per patient)
  - section split            
  - spaCy + negspaCy          
  - uncertainty regex         
  - label                     
```

---
## ⭐ Advanced Section

*For those who finish early.*

---


### Challenge A: Entity extraction

Beyond a binary label we often want structured facts:
- **Laterality**: left/right/bilateral
- **Bone / site**: distal radius, scaphoid, metacarpal, phalanx, …
- **Displacement**: displaced/undisplaced/comminuted

In [ ]:
#@title Click to reveal code
LATERALITY   = re.compile(r'\b(left|right|bilateral|lt|rt)\b', re.IGNORECASE)
BONE         = re.compile(
    r'\b(distal radius|scaphoid|metacarpal|phalanx|phalangeal|'
    r'radial styloid|ulnar styloid|triquetral|lunate)\b', re.IGNORECASE
)
DISPLACEMENT = re.compile(
    r'\b((?:minimally |mildly |markedly )?displaced|undisplaced|nondisplaced|'
    r'comminuted|intra-articular|extra-articular)\b', re.IGNORECASE
)

def extract_entities(row):
    text = str(row.get('header', '')) + ' ' + str(row.get('findings', ''))
    return {
        'laterality'  : sorted(set(m.group().lower() for m in LATERALITY.finditer(text))),
        'bones'       : sorted(set(m.group().lower() for m in BONE.finditer(text))),
        'displacement': sorted(set(m.group().lower() for m in DISPLACEMENT.finditer(text))),
    }

for _, row in df[df['classification'] == 'positive'].head(6).iterrows():
    print(f"{row['fake_patient_id']}: {extract_entities(row)}")

---
### Challenge B: Temporality

*"Previous fracture, now healed"* is not the current injury.

Modify `spacy_has_fracture()` to also skip fracture mentions preceded by
temporal qualifiers like *previous, prior, old, healed, chronic, remote*.

In [ ]:
#@title Click to reveal code
HISTORICAL = re.compile(
    r'\b(previous|prior|old|healed|chronic|remote|known|longstanding)\b',
    re.IGNORECASE
)

def spacy_has_fracture_v2(findings: str) -> bool:
    if not isinstance(findings, str):
        return False
    doc = nlp(findings)
    for ent in doc.ents:
        if ent.label_ != 'FINDING' or ent._.negex:
            continue
        window = findings[max(0, ent.start_char - 60): ent.start_char]
        if HISTORICAL.search(window):
            continue
        return True
    return False


test_cases = [
    ('Previous distal radius fracture, well healed. No new acute fracture.', False),
    ('Acute nondisplaced fracture of the scaphoid waist.',                   True),
    ('No acute fracture. Known old fracture of the radial styloid.',         False),
    ('No acute fracture seen. Boxer fracture of the fifth metacarpal neck.', True),
]

for text, expected in test_cases:
    result = spacy_has_fracture_v2(text)
    status = 'OK  ' if result == expected else 'FAIL'
    print(f'{status}  expected={expected}  got={result}   "{text[:70]}"')

---
### Challenge C: scispaCy 

`en_core_web_sm` is a general English model. For clinical text, **scispaCy** provides
models trained on biomedical literature with better handling of medical abbreviations.

```bash
pip install scispacy
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
```

Try replacing `spacy.load('en_core_web_sm')` with `spacy.load('en_core_sci_sm')` and
compare sentence splitting.

In [ ]:
#@title Click to reveal code
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', ,'spacy==3.7.4','scispacy==0.5.4'], check=True)
model_url = "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', model_url], check=True)
nlp_sci = spacy.load('en_core_sci_sm')

sample_text = df.loc[0, 'findings'] 
doc = nlp(sample_text) 
doc_sci = nlp_sci(sample_text)

print()
print('=== Sentences ===')
for sent in doc.sents:
    print(' ', sent.text)

print('=== Sentences (scispaCy)===')
for sent in doc_sci.sents:
    print(' ', sent.text)
